# Практика 15. Критерії узгодженості й таблиці спряженості: критерій хі-квадрат

**Варіант 1: Київ**

Мета: побудувати таблицю спряженості для сезону й відхилення температури від норми, перевірити їхню незалежність і застосувати критерій узгодженості до категоріальних розподілів.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

variant = 1
city = 'Київ'
base_temp = 9.5
amplitude = 14.0
alpha = 0.05

np.random.seed(42)

def season(month):
    if month in (12, 1, 2):
        return 'зима'
    if month in (3, 4, 5):
        return 'весна'
    if month in (6, 7, 8):
        return 'літо'
    return 'осінь'

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temperature = round(base_temp + seasonal + noise, 1)
        diff = temperature - base_temp
        if diff < -3:
            norm_category = 'холодніше'
        elif diff > 3:
            norm_category = 'тепліше'
        else:
            norm_category = 'звичайно'
        rows.append({
            'місто': city,
            'рік': year,
            'місяць': month,
            'температура': temperature,
            'сезон': season(month),
            'відхилення_від_норми': norm_category,
        })

climate = pd.DataFrame(rows)
season_order = ['зима', 'весна', 'літо', 'осінь']
category_order = ['холодніше', 'звичайно', 'тепліше']

print(f'Варіант {variant}: {city}')
print(f'Розмір набору: {climate.shape[0]} рядків')
print(climate.head())

Варіант 1: Київ
Розмір набору: 48 рядків
  місто   рік  місяць  температура  сезон відхилення_від_норми
0  Київ  2021       1         -4.0   зима            холодніше
1  Київ  2021       2         -2.8   зима            холодніше
2  Київ  2021       3          3.1  весна            холодніше
3  Київ  2021       4         11.0  весна             звичайно
4  Київ  2021       5         16.3  весна              тепліше


## Побудова кліматичного набору

Варіант 1 — Київ: середньорічна температура `9.5 °C`, сезонна амплітуда `14 °C`. Для відтворюваності використано `np.random.seed(42)`.

In [2]:
table = pd.crosstab(
    climate['сезон'],
    climate['відхилення_від_норми']
).reindex(index=season_order, columns=category_order, fill_value=0)
normalized_table = pd.crosstab(
    climate['сезон'],
    climate['відхилення_від_норми'],
    normalize='index'
).reindex(index=season_order, columns=category_order, fill_value=0)

print('Фактична таблиця:')
print(table)
print('\nНормалізована за рядками таблиця:')
print(normalized_table.round(3))
print('\nНайчастіші комбінації:')
print(table.stack()[table.stack() == table.stack().max()])

Фактична таблиця:
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
зима                         12         0        0
весна                         4         4        4
літо                          0         0       12
осінь                         4         4        4

Нормалізована за рядками таблиця:
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
зима                      1.000     0.000    0.000
весна                     0.333     0.333    0.333
літо                      0.000     0.000    1.000
осінь                     0.333     0.333    0.333

Найчастіші комбінації:
сезон  відхилення_від_норми
зима   холодніше               12
літо   тепліше                 12
dtype: int64


## Завдання 1. Таблиця спряженості власного варіанта

Будуємо таблицю абсолютних частот і таблицю часток у кожному рядку. Рядкова нормалізація робить суми рядків рівними 1, тому сезони можна порівнювати незалежно від їхнього розміру.

In [3]:
chi2, p_independence, dof, expected = chi2_contingency(table)
expected_table = pd.DataFrame(expected, index=table.index, columns=table.columns)
residuals = (table - expected_table).abs()
max_cell = residuals.stack().idxmax()

print(f'χ² = {chi2:.4f}, p-value = {p_independence:.8f}, dof = {dof}')
print('Очікувані частоти:')
print(expected_table.round(2))
print(f'Найбільша абсолютна розбіжність: клітинка {max_cell}, {residuals.loc[max_cell]:.2f}')
print('Висновок: відхиляємо H₀, сезон і категорія пов\'язані.' if p_independence < alpha else 'Висновок: не відхиляємо H₀.')

χ² = 38.4000, p-value = 0.00000094, dof = 6
Очікувані частоти:
відхилення_від_норми  холодніше  звичайно  тепліше
сезон                                             
зима                        5.0       2.0      5.0
весна                       5.0       2.0      5.0
літо                        5.0       2.0      5.0
осінь                       5.0       2.0      5.0
Найбільша абсолютна розбіжність: клітинка ('зима', 'холодніше'), 7.00
Висновок: відхиляємо H₀, сезон і категорія пов'язані.


## Завдання 2. Критерій незалежності

H₀: сезон і категорія відхилення від норми незалежні. H₁: між ними є зв'язок. Порівнюємо фактичні та очікувані частоти, отримані за припущенням незалежності.

In [4]:
small_expected = expected.min() < 5
print(f'Мінімальна очікувана частота у початковій таблиці: {expected.min():.1f}')
print(f'Є клітинки з expected < 5? {small_expected}')

merged = pd.DataFrame({
    'не тепліше': table['холодніше'] + table['звичайно'],
    'тепліше': table['тепліше'],
}, index=table.index)
chi2_merged, p_merged, dof_merged, expected_merged = chi2_contingency(merged)
print('\nОб\'єднана таблиця:')
print(merged)
print('Об\'єднані expected:')
print(pd.DataFrame(expected_merged, index=merged.index, columns=merged.columns).round(2))
print(f'χ² = {chi2_merged:.4f}, p-value = {p_merged:.8f}, dof = {dof_merged}')
print('Висновок після об\'єднання: сезон і категорія залишаються пов\'язаними.' if p_merged < alpha else 'Висновок після об\'єднання: підстав відхиляти незалежність недостатньо.')

Мінімальна очікувана частота у початковій таблиці: 2.0
Є клітинки з expected < 5? True

Об'єднана таблиця:
       не тепліше  тепліше
сезон                     
зима           12        0
весна           8        4
літо            0       12
осінь           8        4
Об'єднані expected:
       не тепліше  тепліше
сезон                     
зима          7.0      5.0
весна         7.0      5.0
літо          7.0      5.0
осінь         7.0      5.0
χ² = 26.0571, p-value = 0.00000928, dof = 3
Висновок після об'єднання: сезон і категорія залишаються пов'язаними.


## Завдання 3. Перевірка умови застосовності

У початковій таблиці очікувані частоти для категорії `звичайно` менші за 5. Об'єднуємо `холодніше` та `звичайно` у змістовну категорію `не тепліше`, після чого повторюємо тест.

In [5]:
season_order = ['зима', 'весна', 'літо', 'осінь']
season_counts = climate['сезон'].value_counts().reindex(season_order, fill_value=0)
fit_season = chisquare(f_obs=season_counts.to_numpy(), f_exp=np.full(4, len(climate) / 4))

print('Частоти сезонів:', season_counts.to_dict())
print(f'χ² = {fit_season.statistic:.4f}, p-value = {fit_season.pvalue:.4f}')
print('Висновок: не відхиляємо H₀; рівномірна пропорція узгоджується з побудовою набору.' if fit_season.pvalue >= alpha else 'Висновок: відхиляємо H₀.')

Частоти сезонів: {'зима': 12, 'весна': 12, 'літо': 12, 'осінь': 12}
χ² = 0.0000, p-value = 1.0000
Висновок: не відхиляємо H₀; рівномірна пропорція узгоджується з побудовою набору.


## Завдання 4. Критерій узгодженості для сезонів

У кожному сезоні є по 3 місяці з 12, тому за 48 рядків очікуємо по 12 спостережень на сезон. Перевіряємо H₀ про рівномірний розподіл `25% / 25% / 25% / 25%`.

In [6]:
category_order = ['холодніше', 'звичайно', 'тепліше']
observed_categories = climate['відхилення_від_норми'].value_counts().reindex(category_order, fill_value=0)
expected_categories = np.array([0.20, 0.60, 0.20]) * len(climate)
fit_category = chisquare(f_obs=observed_categories.to_numpy(), f_exp=expected_categories)

print('Фактичні частоти:', observed_categories.to_dict())
print(f'Очікувані частоти за H₀: {expected_categories}')
print(f'χ² = {fit_category.statistic:.4f}, p-value = {fit_category.pvalue:.8f}')
print('Висновок: відхиляємо H₀.' if fit_category.pvalue < alpha else 'Висновок: не відхиляємо H₀.')

Фактичні частоти: {'холодніше': 20, 'звичайно': 8, 'тепліше': 20}
Очікувані частоти за H₀: [ 9.6 28.8  9.6]
χ² = 37.5556, p-value = 0.00000001
Висновок: відхиляємо H₀.


## Завдання 5. Власна заявлена пропорція

**H₀:** категорії `холодніше`, `звичайно`, `тепліше` мають пропорції `20%`, `60%`, `20%` відповідно. Це обґрунтоване припущення як симетрична модель, у якій більшість місяців є близькими до норми, а крайні відхилення в обидва боки трапляються рідше.

## Контрольні питання

**1. Механіка χ².** Статистика `χ² = Σ (observed - expected)² / expected` використовує квадрат, щоб додатні й від'ємні відхилення не взаємознищувалися, а ділення на очікувану частоту враховує масштаб клітинки. Однакова абсолютна різниця важливіша для малої очікуваної частоти, ніж для великої.

**2. Очікувана частота.** За незалежності очікувана частота клітинки дорівнює `частота рядка × частота стовпця / загальна сума`. Це кількість, яку прогнозує добуток маргінальних часток, якщо належність до рядка не змінює розподіл стовпців.

**3. Велике p-value.** Воно означає лише відсутність достатніх підстав відхилити H₀ про незалежність. Це не доведення того, що змінні точно незалежні.

**4. Очікувана частота < 5.** Апроксимація розподілом χ² може бути неточною. Варто об'єднати змістовно близькі категорії, збільшити вибірку або застосувати точніший метод.

## Підсумок

- Для Києва згенеровано 48 місячних спостережень за 2021–2024 роки.
- Побудовано таблиці спряженості для сезону й категорії відхилення від норми.
- Перевірено незалежність через `chi2_contingency()` та умову очікуваних частот.
- Критерієм `chisquare()` перевірено рівномірність сезонів і власну пропорцію категорій.